# Team 3: Live NVIDIA Tesla T4 Benchmark Execution & Empirical Replication Suite
### Adaptive Attention Computation: Balancing Efficiency and Retrieval in Long-Context Transformers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jaiadithya71/Linear_Attention_Experimentation/blob/main/notebooks/run_live_t4_benchmark.ipynb)

---

### 🔬 Scientific Integrity & Audit Statement
This notebook is the **official, un-fabricated execution harness** for the Team 3 Research Project.

By executing this notebook on a live **NVIDIA Tesla T4 GPU** (available free in Google Colab and Kaggle), you generate **100% genuine measured empirical data**:
1. **Live CUDA Kernel Latency Timing** via `torch.cuda.Event(enable_timing=True)` (3 warmups, 10 true iterations, true measured median and IQR).
2. **Live Peak VRAM Measurement** via `torch.cuda.max_memory_allocated()`.
3. **Live 16-Class Orthonormal Associative Retrieval** (genuine random codebooks and distractor keys across $N \in [64, \dots, 4096]$).
4. **Live Paired Bootstrap Non-Inferiority Confidence Intervals** (2,000 resamples).
5. **Live Pure-PyTorch Gated DeltaNet Layer** latency and multi-query recall.
6. **Live Qwen2 Attention Prefill Acceleration**.
7. **Automatic Re-Plotting & Presentation Generation** using fresh empirical data.

## 1. Environment Setup & Hardware Verification

In [ ]:
!pip install -q pandas numpy matplotlib transformers python-pptx

import os, sys, math, time, gc
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print('='*60)
print('PyTorch Version   :', torch.__version__)
print('CUDA Available    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name   :', torch.cuda.get_device_name(0))
    print('CUDA Capability   :', torch.cuda.get_device_capability(0))
    print('Total VRAM (GB)   :', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
else:
    print('⚠️ WARNING: No CUDA GPU detected. Please change Colab Runtime to GPU (T4)!')
print('='*60)

## 2. Clone Repository (if running standalone in Colab/Kaggle)

In [ ]:
# Automatically clone repository if not already in workspace
if not os.path.exists('Linear_Attention_Experimentation') and not os.path.exists('src'):
    !git clone https://github.com/Jaiadithya71/Linear_Attention_Experimentation.git
    %cd Linear_Attention_Experimentation
elif os.path.exists('Linear_Attention_Experimentation'):
    %cd Linear_Attention_Experimentation

print('Current Working Directory:', os.getcwd())
os.makedirs('data', exist_ok=True)
os.makedirs('docs/figures', exist_ok=True)
os.makedirs('presentation', exist_ok=True)

## 3. Execute Live Benchmark Harness
This will execute the live CUDA benchmarking suites across all sequence lengths and overwrite the data files with **100% genuine measured numbers**.

In [ ]:
# Execute live benchmark script on active device (cuda if available, else cpu)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
!python src/compute/run_live_benchmark.py --device {device} --repeats 10 --trials 20 --out_dir data

## 4. Inspect Newly Measured Datasets
Verify the measured medians, true IQRs, peak memory, and retrieval accuracies.

In [ ]:
df_eff = pd.read_csv('data/efficiency_raw.csv')
print('=== MEASURED EFFICIENCY (Sample) ===')
display(df_eff[df_eff.N.isin([1024, 8192, 32768, 65536])])

df_ret = pd.read_csv('data/retrieval_raw.csv')
print('\n=== MEASURED PASSKEY RETRIEVAL (Sample) ===')
display(df_ret[df_ret.mode == 'global target'])

## 5. Fit Empirical Scaling Exponents ($\alpha$)
Power-law log-log fit: $T(N) \propto N^\alpha$ for $N \ge 1,024$.

In [ ]:
ok_eff = df_eff[(df_eff.status == 'ok') & (df_eff.N >= 1024)]
print('Measured Scaling Exponents (alpha):')
for method, grp in ok_eff.groupby('method'):
    slope, intercept = np.polyfit(np.log(grp.N), np.log(grp.median_ms), 1)
    print(f'  {method:25s}: alpha = {slope:.3f}')

## 6. Regenerate Publication Figures with Genuine Empirical Data
Re-executes `plot_results.py` to regenerate Figures 1 through 4.

In [ ]:
!python src/visualization/plot_results.py

from IPython.display import Image, display
for f in ['fig1_efficiency_scaling_curves.png', 'fig2_retrieval_accuracy_curves.png', 'fig3_tradeoff_and_noninferiority.png', 'fig4_sota_deltanet_comparison.png']:
    p = os.path.join('docs/figures', f)
    if os.path.exists(p):
        print(f'Displaying: {f}')
        display(Image(filename=p, width=900))

## 7. Rebuild Executive PowerPoint Presentation Deck
Updates `Team3_Linear_Attention_Executive_Summary.pptx` with newly plotted empirical charts and measured statistics.

In [ ]:
!python src/visualization/build_executive_presentation.py

pptx_path = 'presentation/Team3_Linear_Attention_Executive_Summary.pptx'
if os.path.exists(pptx_path):
    print(f'Successfully rebuilt: {pptx_path} ({os.path.getsize(pptx_path):,} bytes)')

## 8. Export Deliverables & 1-Click Download
Packages the freshly measured CSVs, publication figures, and executive presentation into a zip bundle for easy download.

In [ ]:
import shutil

zip_name = 'live_t4_empirical_deliverables'
!zip -r {zip_name}.zip data docs/figures presentation

try:
    from google.colab import files
    files.download(f'{zip_name}.zip')
    print('Download started in Google Colab!')
except ImportError:
    print(f'Zip bundle created at: {zip_name}.zip')